In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import requests
import re

In [ ]:
login_url='https://www.jobplanet.co.kr/users/sign_in'

email = '####본인계정아이디####'
password = "####본인계정비밀번호####"

LOGIN_INFO = {
    'user[email]' : email,
    'user[password]' : password,
    'commit' : '로그인'
}

session = requests.session()

# 로그인 정보(비밀번호 포함)를 전송하므로 인증서 검증을 끄면 안 됩니다.
res = session.post(login_url, data = LOGIN_INFO)

res.raise_for_status()

In [4]:
# 총 기업 개수 반환하는 함수
def company_count():
  url = 'https://www.jobplanet.co.kr/companies?industry_id=100'
  r = session.get(url)
  r.raise_for_status()

  soup = BeautifulSoup(r.text, 'html.parser')
  #print(soup)
  last_num_html = soup.find('span', {'class': 'num'})
  last_num = last_num_html.get_text()
  return int(last_num)

In [5]:
# 페이지 수 반환하는 함수
def last_page(x):
  if x % 10 == 0:
    last_page_num = x // 10
  else :
    last_page_num = (x // 10) + 1

  return last_page_num

In [6]:
# 마지막 페이지
company_num = company_count()
last_page = last_page(company_num)
print(last_page)

148


In [7]:
number = []
company_name = []
links = []


for k in range(1, last_page+1):
  url = 'https://www.jobplanet.co.kr/companies?&industry_id=100&page=' + str(k)
  r1 = session.get(url)
  r1.raise_for_status()


  soup1 = BeautifulSoup(r1.text, 'html.parser')

  for i in soup1.find_all('dt', attrs={'class': 'us_titb_l3'}):
    for j in i.find_all('a'):
      number_match = re.search(r'/companies/(\d+)/', j['href']) # search 구문 공부 숫자 뽑아내는 코드
      if number_match:
          number.append(number_match.group(1))
      company_name.append(j.text.strip())

      review_url = 'https://www.jobplanet.co.kr/companies/' + number_match.group(1) + '/interviews/' + j.text.strip()
      links.append(review_url)

print(number)
print(company_name)
print(links)

['70749', '3616', '78632', '51105', '87279', '49174', '71990', '89310', '68269', '13055', '53841', '87274', '23183', '47377', '91680', '308150', '65658', '88136', '86465', '87010', '7124', '13535', '92793', '94969', '7502', '56827', '86726', '1973', '87378', '77746', '87555', '70064', '86851', '89009', '63215', '88148', '89731', '93573', '77254', '90449', '92922', '87453', '86725', '89206', '59120', '2927', '44165', '94498', '90129', '87570', '88970', '52165', '274320', '88192', '88193', '91671', '90544', '30128', '47379', '309523', '87272', '82315', '87480', '88335', '78989', '2407', '86753', '89364', '343803', '89546', '86810', '87675', '89354', '35673', '89090', '52676', '91386', '87440', '349160', '93480', '7473', '47536', '93026', '7388', '91398', '345400', '85983', '68602', '94210', '84079', '90644', '87134', '87368', '340281', '52133', '90275', '88063', '97092', '88469', '87563', '94512', '238832', '87147', '310162', '88865', '87307', '243674', '95082', '62154', '87145', '229093

In [8]:
# 회사이름, 링크, 넘버 리스트 --> 데이터 프레임으로 변환하고 csv 파일로 저장하기
df_company_name = pd.DataFrame(company_name)
df_number = pd.DataFrame(number)
df_links = pd.DataFrame(links)

In [11]:
#df_company_name.to_csv('company_name.csv')
#df_number.to_csv('number_f.csv', index = False, encoding = 'utf-8-sig')
#df_links.to_csv('links_f.csv', index = False, encoding = 'utf-8-sig')
#df_company_name.to_csv('company_name_f.csv', index=False, encoding = 'utf-8-sig')

In [22]:
# 상세페이지 - 페이지 수 확인하는 함수 (기업에 해당하는 url 넘겨주기)
def reviews_count(url):
  r = session.get(url)
  r.raise_for_status()

  soup = BeautifulSoup(r.text, 'html.parser')
  review = soup.find('div', {'id': 'viewInterviewsTitle'}).find('span', {'class': 'num'})

  num = review.get_text()
  num = int(num.replace(',', ''))
  return num

In [23]:
# 페이지 계산하는 함수
def reviews_page_count(x):
  if x % 5 == 0:
    reviews_page_num = x // 5
  else:
    reviews_page_num = (x // 5) + 1

  return reviews_page_num

In [24]:
# 면접시기, 느낌 빈 값 처리 함수
def insert_to_list(original_list, index):
    
    tmp_list = []
    add_list = [[company_name, "N"]]
    while len(original_list) > index:
        tmp_list.append(original_list.pop())

    original_list.extend([['N']])
    
    original_list.extend(tmp_list[::-1])
    
    return original_list

In [25]:
len(links)

1479

In [69]:
Date_list = []
career_list = []
edu_list = []
question_list = []
feel_list = []
Dates_list = []
Feel_list = []

# 리뷰 처음 체이지
for href in links[0:21]: # 21번째 다시 해야함 [0:22]로 
  url = href # 특정 회사 페이지 도착
  review_num = reviews_count(url)
  review_page_num = reviews_page_count(review_num)

  for i in range(1, review_page_num+1):
    urld = url + '?page=' + str(i)
    print(urld)

    rew = session.get(urld, headers={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'})
    rew.raise_for_status()

    soup1 = BeautifulSoup(rew.text, 'html.parser')

    company_name = re.search(r'interviews/([^/]+)',url).group(1)


    for i in soup1.find_all("dl", class_="ctbody_lft"):
      for kk in i.find_all("dd", class_="txt1"):
        date = kk.get_text().strip()

        Date_list.append([company_name, date])

    for i in soup1.find_all("dl", class_="ctbody_lft"):
      k = i.find_all("dd", class_="txt1")

      Dates_list.append([company_name, k])

    # 중간에 기업 댓글 달리면 오류나서 코드 수정
    
    for p in soup1.find_all('div', {'class' : 'content_top_ty2'}):
      for k in p.find_all('span', {'class' : 'txt1'}):
        careers = k.get_text().strip().split('\n')[0]
        career_list.append([company_name, careers])

    for p in soup1.find_all('div', {'class' : 'content_top_ty2'}):
      for i in p.find_all('span', {'class' : 'txt1'}):
        edu1 = p.get_text().strip().split('\n')[2].strip()
        edu_list.append([company_name, edu1])

    for p in soup1.find_all('span', {'class': 'answer mobile_full_content notranslate'}):
      feel = p.get_text()
      
      feel_list.append((company_name, feel))

    for i in soup1.find_all("dl", class_="tc_list"):
      col = i.find_all('span', {'class': 'answer mobile_full_content notranslate'})

      Feel_list.append([company_name, col])

  
    for i in soup1.find_all("dl", class_="tc_list"):
      o = i.find("dd", class_="df1").get_text().strip().split('\n')

      question_list.append([company_name, o])



https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=1
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=2
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=3
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=4
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=5
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=6
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=7
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=8
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=9
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=10
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=11
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=12
https://www.jobplanet.co.kr/companies/70749/interviews/(주)에스씨케이컴퍼니?page=13
https://www.jobplanet.co.kr/compan

In [70]:
# 면접 일자와 느낌 빈 값 확인하도록 비교 리스트 생성, 빈 값 인덱스 추출
for i in range(len(Dates_list)):
   if Dates_list[i][1] == []:
      Dates_list[i][1] = ['noday']

b = [p[1] for p in Dates_list]

rest_list = list(filter(lambda x: b[x] == ['noday'], range(len(b))))


for i in range(len(Feel_list)):
   if Feel_list[i][1] == []:
      Feel_list[i][1] = ['no']

c = [q[1] for q in Feel_list]

Rest_list = list(filter(lambda x: c[x] == ['no'], range(len(c))))

In [71]:
# 빈 값 인덱스에 맞춰 넣기
for i in rest_list:
    insert_to_list(Date_list, i)

for j in Rest_list:
    insert_to_list(feel_list, j)

In [72]:
Date_df = pd.DataFrame(Date_list, columns=['company_name','date'])
career_df = pd.DataFrame(career_list, columns=['company_name', 'career'])
edu_df = pd.DataFrame(edu_list, columns=['company_name', 'edu'])
question_df = pd.DataFrame(question_list, columns=['company_name', 'question'])
feel_df = pd.DataFrame(feel_list, columns=['company_name', 'feel'])
Dates_df = pd.DataFrame(Dates_list, columns=['company_name','DATE'])
Feel_df = pd.DataFrame(Feel_list, columns=['company_name', 'Feel'])


df1 = pd.concat([career_df, Date_df['date'], edu_df['edu'], question_df['question'], feel_df['feel']], axis=1)
df1['feel'] = df1['feel'].str.replace('\n', '').str.strip().str.replace('\r', ' ').str.replace('-', ' ').str.replace('=', '').str.strip()
df1['question'] = df1['question'].str[0].str.strip('[]').str.replace('\r', ' ').str.replace('-', ' ').str.replace('=', '').str.strip()

df1

,company_name,career,date,edu,question,feel
0,(주)에스씨케이컴퍼니,서비스/고객지원,2020/03,사원-전문대졸,1.얼마나 오래 근무할건지 2. 스타벅스에 왜 입사하고 싶은지 3. 다른 카페와 스...,1. 진급생각 있다 2. 대표적인 프차라 큰 곳에서 배우고 싶었다 3. 다른카페가면...
1,(주)에스씨케이컴퍼니,서비스/고객지원,2018/01,사원-대졸,스타벅스 이용 경험 여부 스타벅스 이용 시 느꼈던 점 본인은 어떤 사람인가?,"이용 경험 있음 단순 음료 판매점이 아닌, 경험과 가치를 제공하는 곳이라 생각 ..."
2,(주)에스씨케이컴퍼니,서비스/고객지원,2021/08,사원-대졸,주로 일 어떻게 배우고 나아가는지 등 입사 후의 이야기를 많이 해주셨다,굉장히 친절하고 잘 안내해주시고 분위기를 편하게 해주시려고 노력하셨던 것 같다!
3,(주)에스씨케이컴퍼니,서비스/고객지원,2018/04,기타,"스타벅스가 제3의 공간이라고 하는데 왜 그렇게 생각하는지?, 매장 방문해 본 적 있...",대체적으로 친절한 편이었음
4,(주)에스씨케이컴퍼니,서비스/고객지원,2020/09,사원-대졸,1. 1분 자기소개 말고 스스로의 진솔한 이야기를 듣고 싶다 2. 점장 부재 시 업...,"사내 전환 면접이니만큼 확실한 내정자가 정해져 있고, 평가가 모호한 지원자에 대해 ..."
...,...,...,...,...,...,...
13555,코웨이(주),기획/경영,2013/05,사원-대졸,코웨이 혁신성장을 위한 아이디어를 질문. 자신들도 잘 모르겠는 부분들에 대해서 너무...,현재 시행하고 있는 사원 아이디어 공모 관련해서.. 평이하게 대답 함..
13556,코웨이(주),생산/제조,2013/11,사원-대졸,다른 사람과 관계가 안좋을때 본인만의 극복 방법이 무엇인가?,None
13557,코웨이(주),기타,2013/11,사원-대졸,"면접 주제는 어렵지 않았고, 코웨이란 무엇이냐고 생각하는 것이 마지막 질문이었습니다...",None
13558,코웨이(주),영업/제휴,2014/05,기타,"영업으로 지원하게 된 계기, 자신의 경력에 대해 설명하시오.",자기만의 이야기를 일목요연하게 설명하면 되겠음.


In [20]:
# 인덱스 포함 csv 파일
df.to_csv('mk_interviews_index4.csv', encoding='utf-8-sig', escapechar='/')

In [73]:
# 인덱스 미포함 csv 파일
df1.to_csv('mk_interviews_no_index1.csv', index=False, encoding='utf-8-sig', escapechar='/')